In [45]:
import os, json, grpc, pandas as pd
from pathlib import Path
from senzing import SzEngineFlags as F
from senzing_grpc import SzAbstractFactoryGrpc
from sqlalchemy import create_engine
from pprint import pprint
from IPython.display import JSON

channel = grpc.insecure_channel(
    f"{os.getenv('SENZING_GRPC_HOST', 'resolver')}:8261",
    options=[('grpc.max_receive_message_length', 200 * 1024 * 1024)])
grpc.channel_ready_future(channel).result(timeout=20)

sz = SzAbstractFactoryGrpc(channel)
engine = sz.create_engine()
print("✅ connected to the main database")

✅ connected to the main database


In [15]:
DATA = Path('/workspace/data/filtered')
DELIM = {'.tsv': 'tab', '.csv': 'comma'}
PIPE = {'Agents.csv', 'Filings.csv', 'Principals.csv'}

for f in sorted(DATA.iterdir()):
    rows = sum(1 for _ in f.open(encoding='utf-8-sig', errors='replace')) - 1
    size = f.stat().st_size
    human = f"{size/1e6:.1f} MB" if size >= 1e6 else f"{size/1e3:.0f} KB"
    sep = 'pipe (*|*)' if f.name in PIPE else DELIM[f.suffix]
    print(f"  {f.name:<52} {rows:>7,} rows  {human:>8}  {sep}")

  Agents.csv                                             7,421 rows    1.2 MB  pipe (*|*)
  CA-ABC-LicenseReport-nonretail.csv                        19 rows      3 KB  comma
  CA-ABC-LicenseReport-retail.csv                          201 rows     30 KB  comma
  Filings.csv                                            7,801 rows    2.9 MB  pipe (*|*)
  Principals.csv                                         5,701 rows    858 KB  pipe (*|*)
  burlingame.tsv                                         4,781 rows    448 KB  tab
  master-list-of-ca-licensed-contractors-personnel.csv  10,215 rows    2.4 MB  comma
  master-list-of-ca-licensed-contractors.csv             3,274 rows    1.1 MB  comma


In [17]:
burlingame = pd.read_csv(DATA/'burlingame.tsv', sep='\t', dtype=str)
burlingame.head()

,Account #,Business Name,Start Date,Expire Date,Address
0,20509315,101 MOBILITY,7/1/2025,6/30/2027,"1731 ADRIAN RD STE 9, BURLINGAME, CA 94010-2109"
1,20510000,"1120 BROADWAY, LLC",10/28/2024,6/30/2027,"1120 BROADWAY, BURLINGAME, CA 94010-3422"
2,20501122,1199 HOWARD LLC,7/1/2010,6/30/2027,"330 PRIMROSE RD # 606, BURLINGAME, CA 94010-4032"
3,20501289,1202 HAIR STUDIO (GINA A GIUSTI),11/5/2010,6/30/2026,"1202 CAPUCHINO AVE, BURLINGAME, CA 94010-3404"
4,20504964,1202 HAIR STUDIO - HAIR BY MARINA,2/1/2022,6/30/2027,"1202 CAPUCHINO, BURLINGAME, CA 94010"


In [19]:
# the three state filing files use '*|*' as their delimiter
agents = pd.read_csv(DATA/'Agents.csv', sep=r'\*\|\*', engine='python', dtype=str)
agents.head()
#agents[['ENTITY_NAME', 'ORG_NAME', 'FIRST_NAME', 'LAST_NAME', 'PHYSICAL_CITY']].head()

,ENTITY_NAME,ENTITY_NUM,ORG_NAME,FIRST_NAME,MIDDLE_NAME,LAST_NAME,PHYSICAL_ADDRESS1,PHYSICAL_ADDRESS2,PHYSICAL_ADDRESS3,PHYSICAL_CITY,PHYSICAL_STATE,PHYSICAL_COUNTRY,PHYSICAL_POSTAL_CODE,AGENT_TYPE
0,Busse Manufacturing LLC,B20260056953,NaN,Richard,Hugh,Busse,610 TRAYER AVE,NaN,NaN,GLENDORA,CA,UNITED STATES,91741,Individual Agent
1,Brilliant Day Consulting LLC,B20260116027,NaN,Mingrui,NaN,Xiong,701 S WOODS AVE,NaN,NaN,FULLERTON,CA,UNITED STATES,92832,Individual Agent
2,FEROCIOUS PLUMBING INC,B20260200247,NaN,ANGEL,NaN,OCEGUEDA,3360 SOUTHERN AVE,NaN,NaN,SOUTH GATE,CA,UNITED STATES,90280,Individual Agent
3,"Get Hired, Properly! LLC",B20260116314,NaN,Mingrui,NaN,Xiong,701 S WOODS AVE,NaN,NaN,FULLERTON,CA,UNITED STATES,92832,Individual Agent
4,Guardian Traffic Control LLC,B20260185536,NaN,Amanda,NaN,Landin,3755 WILDRYE DR,NaN,NaN,SAN BERNARDINO,CA,UNITED STATES,92407,Individual Agent


In [21]:
contractors = pd.read_csv(DATA/'master-list-of-ca-licensed-contractors.csv', dtype=str)

desilva = contractors[contractors.BusinessName.str.contains('DESILVA', case=False, na=False)]
#desilva[['LicenseNo', 'BusinessName', 'MailingAddress', 'BusinessPhone']]
desilva.head(10)

,LicenseNo,LastUpdate,BusinessName,BUS-NAME-2,FullBusinessName,MailingAddress,City,State,County,ZIPCode,...,DBSuretyCompany,DBNumber,DBEffectiveDate,DBCancellationDate,DBAmount,DateRequired,DiscpCaseRegion,DBBondReason,DBCaseNo,NAME-TP-2
600,1078219,04/27/2026,DESILVA GATES CONSTRUCTION LLC,NaN,NaN,11555 DUBLIN BOULEVARD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
685,1087418,04/07/2026,DESILVA GATES - PACIFIC STATES A JOINT VENTURE,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
739,1093103,06/03/2026,DESILVA GATES CONSTRUCTION LLC - MCM A JOINT V...,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
796,1099346,04/07/2026,DESILVAGATES CONSTRUCTION - VANGUARD A JOINT V...,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
836,1103207,04/07/2026,DESILVA GATES-VIKING A JOINT VENTURE,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1042,1125855,07/29/2026,DESILVA GATES - GHILOTTI A JOINT VENTURE,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1358,1156955,06/22/2026,DESILVA GATES - ANVIL BUILDERS A JOINT VENTURE,NaN,NaN,11555 DUBLIN BLVD,DUBLIN,CA,Alameda,94568,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
abc = pd.read_csv(DATA/'CA-ABC-LicenseReport-retail.csv', dtype=str)

#print("City business licence:")
#print(burlingame[burlingame['Business Name'].str.contains('TOPGOLF', na=False)].to_string(index=False))
#print("\nState liquor licence:")
#print(abc[abc['Business Name'].str.contains('TOPGOLF', na=False)]
#      [['License Number','Primary Owner','Premises Addr.','Business Name']].to_string(index=False))

abc[abc['Business Name'].str.contains('TOPGOLF', na=False)].head()

,License Number,Status,License Type,Orig. Iss. Date,Expir. Date,Primary Owner,Premises Addr.,Business Name,Geo Code
169,657740,ACTIVE,47,11/19/2024,10/31/2026,"TOPGOLF USA BURLINGAME, LLC","258 ANZA BLVD,BURLINGAME, CA 94010-1939Census...",TOPGOLF,4103
170,657740,ACTIVE,68,11/19/2024,10/31/2026,"TOPGOLF USA BURLINGAME, LLC","258 ANZA BLVD,BURLINGAME, CA 94010-1939Census...",TOPGOLF,4103


In [24]:
burlingame[burlingame['Business Name'].str.contains('TOPGOLF', na=False)].head()

,Account #,Business Name,Start Date,Expire Date,Address
4377,20508711,TOPGOLF,12/13/2024,6/30/2027,"258 N ANZA BLVD, BURLINGAME, CA 94010-1939"


Notice that the above two have addresses only separated by one letter (Anza vs. N Anza).  But there is no shared identifier between these two.

In [30]:
print(f"Burlingame business licences: {len(burlingame):,}")
print(f"Distinct business names:      {burlingame['Business Name'].nunique():,}")
print(f"Contractor licences:          {len(contractors):,}")
print(f"Distinct business names:      {contractors.BusinessName.nunique():,}")
print(f"Distinct phone numbers:       {contractors.BusinessPhone.nunique():,}")

Burlingame business licences: 4,781
Distinct business names:      4,740
Contractor licences:          3,274
Distinct business names:      3,058
Distinct phone numbers:       3,088


Exact-match deduplication only catches identical strings. Everything above — the missing space, the missing N, the joint ventures, the LLC suffix — slips straight through.

In [31]:
# The raw row from the city file
raw = burlingame[burlingame['Business Name'] == 'TOPGOLF'].iloc[0]
print("RAW ROW:")
print(raw.to_string(), "\n")

# The same row, mapped into Senzing's format and stored
stored = json.loads(engine.get_record('BURLINGAME', '20508711'))
print("AS STORED IN SENZING:")
print(json.dumps(stored['JSON_DATA'], indent=2))

RAW ROW:
Account #                                          20508711
Business Name                                       TOPGOLF
Start Date                                       12/13/2024
Expire Date                                       6/30/2027
Address          258 N ANZA BLVD, BURLINGAME, CA 94010-1939 

AS STORED IN SENZING:
{
  "DATA_SOURCE": "BURLINGAME",
  "RECORD_ID": "20508711",
  "FEATURES": [
    {
      "RECORD_TYPE": "ORGANIZATION"
    },
    {
      "NAME_ORG": "TOPGOLF"
    },
    {
      "ADDR_TYPE": "BUSINESS",
      "ADDR_FULL": "258 N ANZA BLVD, BURLINGAME, CA 94010-1939"
    }
  ],
  "LICENSE_START_DATE": "12/13/2024",
  "LICENSE_EXPIRE_DATE": "6/30/2027"
}


In [32]:
tables = pd.read_sql("""
    SELECT 'dsrc_record'         AS table_name, count(*) FROM dsrc_record
    UNION ALL SELECT 'lib_feat',            count(*) FROM lib_feat
    UNION ALL SELECT 'obs_ent',             count(*) FROM obs_ent
    UNION ALL SELECT 'res_ent',             count(*) FROM res_ent
    UNION ALL SELECT 'res_relate',          count(*) FROM res_relate
    UNION ALL SELECT 'workshop.entity_geo', count(*) FROM workshop.entity_geo
    ORDER BY 2 DESC
""", db)
print(tables.to_string(index=False))

         table_name  count
           lib_feat 498992
        dsrc_record  39413
            obs_ent  33906
            res_ent  31339
         res_relate  11822
workshop.entity_geo   2482


          table_name   count
            lib_feat  498992     <- every name, address, phone it extracted
         dsrc_record   39413     <- your records, kept verbatim
             obs_ent   33906
             res_ent   31339     <- the entities it concluded exist
          res_relate   11822     <- entities that are related but not the same
 workshop.entity_geo    2482     <- coordinates we added afterwards
 

In [33]:
records  = pd.read_sql("SELECT count(*) c FROM dsrc_record", db).c[0]
entities = pd.read_sql("SELECT count(*) c FROM res_ent", db).c[0]
related  = pd.read_sql("SELECT count(*) c FROM res_relate", db).c[0]

print(f"records in:       {records:,}")
print(f"entities out:     {entities:,}")
print(f"records merged:   {records - entities:,}")
print(f"related pairs:    {related:,}")

records in:       39,413
entities out:     31,339
records merged:   8,074
related pairs:    11,822


In [2]:
config = sz.create_configmanager()
cfg = config.create_config_from_config_id(config.get_default_config_id())
names = {d['DSRC_ID']: d['DSRC_CODE']
         for d in json.loads(cfg.get_data_source_registry())['DATA_SOURCES']}

db = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'workshop')}@"
    f"{os.getenv('POSTGRES_HOST', 'postgres')}:5432/"
    f"{os.getenv('POSTGRES_DB', 'erkg')}")

counts = pd.read_sql("SELECT dsrc_id, count(*) AS records FROM dsrc_record "
                     "GROUP BY 1 ORDER BY 2 DESC", db)

counts['data_source'] = counts.dsrc_id.map(names)
print(counts[['data_source', 'records']].to_string(index=False))
print(f"\ntotal: {counts.records.sum():,} records")

            data_source  records
CA-CONTRACTOR-PERSONNEL    10215
         CA-SOS-FILINGS     7801
          CA-SOS-AGENTS     7421
      CA-SOS-PRINCIPALS     5701
             BURLINGAME     4781
         CA-CONTRACTORS     3274
             ABC-RETAIL      201
          ABC-NONRETAIL       19

total: 39,413 records


In [3]:
# Records go IN. Entities come OUT. This is the core idea.
entity = json.loads(
    engine.get_entity_by_record_id('BURLINGAME', '20508711'))['RESOLVED_ENTITY']

print(f"entity {entity['ENTITY_ID']}: {entity['ENTITY_NAME']}\n")
for r in entity['RECORDS']:
    print(f"  {r['DATA_SOURCE']:<16} {r['RECORD_ID']}")

entity 4372: TOPGOLF

  BURLINGAME       20508711
  ABC-RETAIL       657740-47
  ABC-RETAIL       657740-68


In [4]:
for family, values in entity['FEATURES'].items():
    print(f"{family}:")
    for v in values:
        print(f"    {v['FEAT_DESC']}")

ADDRESS:
    258 N ANZA BLVD, BURLINGAME, CA 94010-1939
NAME:
    TOPGOLF
RECORD_TYPE:
    ORGANIZATION


In [5]:
how = json.loads(engine.how_entity_by_entity_id(
    entity['ENTITY_ID'], flags=F.SZ_HOW_ENTITY_DEFAULT_FLAGS))

for step in how['HOW_RESULTS']['RESOLUTION_STEPS']:
    info = step['MATCH_INFO']
    print(f"step {step['STEP']}: {info['MATCH_KEY']}   (rule {info['ERRULE_CODE']})")

step 1: +NAME+ADDRESS   (rule CNAME_CFF_CEXCL)


In [6]:
results = json.loads(engine.search_by_attributes(
    json.dumps({'NAME_ORG': 'TOPGOLF', 'ADDR_CITY': 'BURLINGAME'})))

for hit in results['RESOLVED_ENTITIES']:
    e = hit['ENTITY']['RESOLVED_ENTITY']
    print(f"{e['ENTITY_NAME']}  (entity {e['ENTITY_ID']})  match: {hit['MATCH_INFO']['MATCH_KEY']}")

TOPGOLF  (entity 4372)  match: +NAME


In [7]:
full = json.loads(engine.get_entity_by_record_id(
    'BURLINGAME', '20509774', flags=F.SZ_ENTITY_DEFAULT_FLAGS))

print(f"{full['RESOLVED_ENTITY']['ENTITY_NAME']} is related to:\n")
for r in full['RELATED_ENTITIES'][:6]:
    print(f"  {r['ENTITY_NAME'][:36]:<36} {r['MATCH_LEVEL_CODE']}  {r['MATCH_KEY']}")

TOLENTINO CONSTRUCTION & RESTORATION INC is related to:

  REPUBLIC REGISTERED AGENT INC        POSSIBLY_RELATED  +ADDRESS
  PACIFIC OPERATIONAL LLC              POSSIBLY_RELATED  +ADDRESS
  GOLDEN TERRA GROUP LLC               POSSIBLY_RELATED  +ADDRESS
  DIABLO VALLEY CONSTRUCTION INC       POSSIBLY_RELATED  +ADDRESS
  OW BUILDERS LLC                      POSSIBLY_RELATED  +ADDRESS
  D&G GARDENING LLC                    POSSIBLY_RELATED  +ADDRESS


In [8]:
hub = json.loads(
    engine.get_entity_by_record_id('CA-SOS-AGENTS', 'B20260301695'))['RESOLVED_ENTITY']

print(f"{hub['ENTITY_NAME']}")
print(f"  {len(hub['RECORDS'])} records")
print(f"  {len(hub['FEATURES']['NAME'])} distinct name, {len(hub['FEATURES']['ADDRESS'])} distinct address")

NORTHWEST REGISTERED AGENT INC
  585 records
  1 distinct name, 1 distinct address


In [9]:
rows = []
handle = engine.export_json_entity_report(F.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while (line := engine.fetch_next(handle)):
        e = json.loads(line)['RESOLVED_ENTITY']
        for r in e.get('RECORDS', []):
            rows.append({'entity_id': e['ENTITY_ID'],
                         'data_source': r['DATA_SOURCE'],
                         'match_key': r.get('MATCH_KEY', '')})
finally:
    engine.close_export_report(handle)

df = pd.DataFrame(rows)
print(f"{len(df):,} records -> {df.entity_id.nunique():,} entities")
print(f"\ntop match keys:\n{df[df.match_key != ''].match_key.value_counts().head(10).to_string()}")

39,413 records -> 31,339 entities

top match keys:
match_key
+EXACTLY_SAME                      4192
+NAME+ADDRESS                      3747
+NAME+ADDRESS+REGISTRATION_DATE      61
+NAME+ADDRESS+PHONE                  44
+ADDRESS+PHONE+LIKELY_NAME           13
+NAME+REGISTRATION_DATE               8
+NAME+PHONE                           7


In [11]:
geo = pd.read_sql("""
    SELECT street, city, placekey, latitude, longitude
    FROM workshop.entity_geo WHERE entity_id = %(eid)s
""", db, params={'eid': entity['ENTITY_ID']})

print(geo.to_string(index=False))

shared = pd.read_sql("""
    SELECT placekey, count(DISTINCT entity_id) AS businesses, min(street) AS address
    FROM workshop.entity_geo GROUP BY placekey
    HAVING count(DISTINCT entity_id) > 1 ORDER BY 2 DESC LIMIT 5
""", db)

print(f"\nbuildings with several businesses:\n{shared.to_string(index=False)}")

         street       city               placekey  latitude   longitude
258 N ANZA BLVD BURLINGAME 0bfl7ardgt@5vg-7y5-2ff 37.590386 -122.352376
  258 ANZA BLVD BURLINGAME 0bfl7ardgt@5vg-7y5-2ff 37.590386 -122.352376

buildings with several businesses:
              placekey  businesses                address
       226@5vg-7y3-b8v          87 533 AIRPORT BLVD # 400
0n4srtfwuu@5vg-7xz-3kf          62            PO BOX 1032
       22k@5vg-7y3-4jv          45     850 BURLINGAME AVE
       22c@5vg-7xx-qvf           9    1410 BURLINGAME AVE
       25x@5vg-7xx-sqz           9 345 LORTON AVE STE 306


In [34]:
# DeSilva Gates: 13 records that a GROUP BY would have called 13 companies
e = json.loads(engine.get_entity_by_record_id(
    'CA-CONTRACTORS', '1087418', flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']

print(f"entity {e['ENTITY_ID']}: {e['ENTITY_NAME']}  ({len(e['RECORDS'])} records)\n")
print("NAMES Senzing decided all belong to this one company:")
for n in e['FEATURES']['NAME']:
    print(f"   {n['FEAT_DESC']}")
print("\nADDRESS:")
for a in e['FEATURES']['ADDRESS']:
    print(f"   {a['FEAT_DESC']}")
print("\nPHONES:")
for p in e['FEATURES']['PHONE']:
    print(f"   {p['FEAT_DESC']}")

entity 1354: DESILVA GATES CONSTRUCTION LLC  (13 records)

NAMES Senzing decided all belong to this one company:
   DE SILVA GATES CONSTRUCTION - MYERS A JOINT VENTURE
   DE SILVA GATES CONSTRUCTION LP
   DESILVA GATES - ANVIL BUILDERS A JOINT VENTURE
   DESILVA GATES - GHILOTTI A JOINT VENTURE
   DESILVA GATES - PACIFIC STATES A JOINT VENTURE
   DESILVA GATES CONSTRUCTION LLC
   DESILVA GATES CONSTRUCTION LLC - MCM A JOINT VENTURE
   PACFIC STATES-TYRRELL A JOINT VENTURE
   PACIFIC STATES - IO ENVIRINMENTAL A JOINT VENTURE
   PACIFIC STATES - SUKUT A JOINT VENTURE

ADDRESS:
   11555 DUBLIN BLVD, DUBLIN, CA 94568-2854

PHONES:
   (925) 361 1533
   (925) 803 4277
   (925) 829 9220


In [35]:
how = json.loads(engine.how_entity_by_entity_id(
    e['ENTITY_ID'], flags=F.SZ_HOW_ENTITY_DEFAULT_FLAGS))

for step in how['HOW_RESULTS']['RESOLUTION_STEPS']:
    print(f"step {step['STEP']}: {step['MATCH_INFO']['MATCH_KEY']}")

step 1: +NAME+ADDRESS
step 2: +LIKELY_NAME+ADDRESS+PHONE
step 3: +NAME+ADDRESS
step 4: +NAME+ADDRESS
step 5: +LIKELY_NAME+ADDRESS+PHONE
step 6: +LIKELY_NAME+ADDRESS+PHONE
step 7: +NAME+ADDRESS+PHONE
step 8: +LIKELY_NAME+ADDRESS+PHONE
step 9: +LIKELY_NAME+ADDRESS+PHONE
step 10: +LIKELY_NAME+ADDRESS+PHONE
step 11: +LIKELY_NAME+ADDRESS+PHONE
step 12: +LIKELY_NAME+ADDRESS+PHONE


In [36]:
briten = json.loads(engine.get_entity_by_record_id(
    'CA-CONTRACTORS', '816198', flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']

print(f"{briten['ENTITY_NAME']}  ({len(briten['RECORDS'])} records)\n")
for family in ('NAME', 'ADDRESS', 'PHONE'):
    print(f"{family}:")
    for v in briten['FEATURES'].get(family, []):
        print(f"   {v['FEAT_DESC']}")

how = json.loads(engine.how_entity_by_entity_id(
    briten['ENTITY_ID'], flags=F.SZ_HOW_ENTITY_DEFAULT_FLAGS))
print("\nsteps:", [s['MATCH_INFO']['MATCH_KEY']
                   for s in how['HOW_RESULTS']['RESOLUTION_STEPS']])

BRITEN FIRE SERVICE INC  (3 records)

NAME:
   BRITEN FIRE SERVICE INC
ADDRESS:
   240 FLOURNOY STREET SAN FRANCISCO CA 94112
   5538 3RD ST, SAN FRANCISCO, CA 94124-2607
PHONE:
   (415) 713 5115

steps: ['+NAME+ADDRESS', '+NAME+PHONE']


In [38]:
# Two entities. Same address, same registration date, names differing by ONE character.
a = json.loads(engine.get_entity_by_record_id(
    'CA-SOS-FILINGS', 'B20260322293', flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']
b = json.loads(engine.get_entity_by_record_id(
    'CA-SOS-FILINGS', 'B20260322300', flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']

for e in (a, b):
    print(f"entity {e['ENTITY_ID']}: {e['ENTITY_NAME']}  ({len(e['RECORDS'])} records)")
    for n in e['FEATURES']['NAME']:
        print(f"    name     {n['FEAT_DESC']}")
    print(f"    address  {e['FEATURES']['ADDRESS'][0]['FEAT_DESC']}")
    print(f"    filed    {e['FEATURES']['REGISTRATION_DATE'][0]['FEAT_DESC']}\n")

entity 400130: 1617-A Chapala Street LLC  (2 records)
    name     1617-A Chapala Street LLC
    name     1621-A Chapala Street LLC
    address  461 SAN YSIDRO ROAD MONTECITO CA 93108 United States
    filed    2026-07-13

entity 400131: 1617-B Chapala Street LLC  (2 records)
    name     1617-B Chapala Street LLC
    name     1621-B Chapala Street LLC
    address  461 SAN YSIDRO ROAD MONTECITO CA 93108 United States
    filed    2026-07-13



In [39]:
why = json.loads(engine.why_entities(
    a['ENTITY_ID'], b['ENTITY_ID'], flags=F.SZ_WHY_ENTITIES_DEFAULT_FLAGS))
info = why['WHY_RESULTS'][0]['MATCH_INFO']

print(f"{info['WHY_KEY']}   ->   {info['MATCH_LEVEL_CODE']}\n")
for family, scores in info['FEATURE_SCORES'].items():
    s = scores[0]
    print(f"  {family:<20} {s['SCORE_BUCKET']}")

+ADDRESS+REGISTRATION_DATE   ->   POSSIBLY_RELATED

  ADDRESS              SAME
  NAME                 PLAUSIBLE
  RECORD_TYPE          SAME
  REGISTRATION_DATE    SAME


Three features identical. The fourth only PLAUSIBLE — and that's enough to refuse the merge. They're POSSIBLY_RELATED, which is Senzing saying these are connected, and I am not going to guess.

Two things make this the best example in the notebook:

Look at what did merge. Entity 400130 contains 1617-A and 1621-A — different street numbers. Entity 400131 has 1617-B and 1621-B. So Senzing merged across a changed number but refused to merge across a changed letter. It read A and B as unit designators deliberately distinguishing sibling entities, while 1617 vs 1621 looked like a data variant. That's judgment, not string distance — any edit-distance rule would do exactly the opposite, since 1617-A/1617-B differ by one character and 1617-A/1621-A differ by two.

This is what a lawyer actually did. Someone set up per-unit LLCs at one address on one day. Merging them would destroy the only thing that matters about them. A dedup script tuned to catch DeSilva Gates' seven name variants would almost certainly collapse these four into one — which is why "just use fuzzy matching" isn't an answer.

In [48]:
print(json.dumps(why, indent=2))

{
  "WHY_RESULTS": [
    {
      "ENTITY_ID": 400130,
      "ENTITY_ID_2": 400131,
      "MATCH_INFO": {
        "WHY_KEY": "+ADDRESS+REGISTRATION_DATE",
        "WHY_ERRULE_CODE": "SFF",
        "MATCH_LEVEL_CODE": "POSSIBLY_RELATED",
        "CANDIDATE_KEYS": {
          "ADDR_KEY": [
            {
              "FEAT_ID": 302032,
              "FEAT_DESC": "461|SN ASTR||MNTST"
            },
            {
              "FEAT_ID": 302033,
              "FEAT_DESC": "461|SN ASTR||93108"
            }
          ],
          "NAMEADDR_KEY": [
            {
              "FEAT_ID": 404710,
              "FEAT_DESC": "STRT|XPL|ADDR_KEY.EXPRESSION=461|SN ASTR||MNTST"
            },
            {
              "FEAT_ID": 404715,
              "FEAT_DESC": "STRT|XPL|ADDR_KEY.EXPRESSION=461|SN ASTR||93108"
            }
          ],
          "NAMEDATE_KEY": [
            {
              "FEAT_ID": 404723,
              "FEAT_DESC": "STRT|XPL|REGISTRATION_DATE=21307"
            },
          

In [49]:
how = json.loads(engine.how_entity_by_entity_id(
    1354, flags=F.SZ_HOW_ENTITY_DEFAULT_FLAGS))['HOW_RESULTS']

# 1. the shape - which steps exist and what drove each
for s in how['RESOLUTION_STEPS']:
    print(s['STEP'], s['MATCH_INFO']['MATCH_KEY'], s['MATCH_INFO']['ERRULE_CODE'])

# 2. one whole step, readable at ~3K chars
print(json.dumps(how['RESOLUTION_STEPS'][1], indent=2))

# 3. just the evidence for that step
print(json.dumps(how['RESOLUTION_STEPS'][1]['MATCH_INFO']['FEATURE_SCORES'], indent=2))

# 4. the end state - small, prints fine
print(json.dumps(how['FINAL_STATE'], indent=2))


1 +NAME+ADDRESS CNAME_CFF_CEXCL
2 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
3 +NAME+ADDRESS CNAME_CFF_CEXCL
4 +NAME+ADDRESS CNAME_CFF_CEXCL
5 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
6 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
7 +NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
8 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
9 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
10 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
11 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
12 +LIKELY_NAME+ADDRESS+PHONE MFF_LNAME_CEXCL
{
  "STEP": 2,
  "VIRTUAL_ENTITY_1": {
    "VIRTUAL_ENTITY_ID": "V600601",
    "MEMBER_RECORDS": [
      {
        "INTERNAL_ID": 600601,
        "RECORDS": [
          {
            "DATA_SOURCE": "CA-CONTRACTORS",
            "RECORD_ID": "1078219"
          }
        ]
      }
    ]
  },
  "VIRTUAL_ENTITY_2": {
    "VIRTUAL_ENTITY_ID": "V600740",
    "MEMBER_RECORDS": [
      {
        "INTERNAL_ID": 600740,
        "RECORDS": [
          {
            "DATA_SOURCE": "CA-CONTRACTORS",
          

In [50]:
step = {k: v for k, v in how['RESOLUTION_STEPS'][1].items() if k != 'MATCH_INFO'}
step['MATCH_INFO'] = {k: v for k, v in how['RESOLUTION_STEPS'][1]['MATCH_INFO'].items()
                      if k != 'CANDIDATE_KEYS'}
print(json.dumps(step, indent=2))

{
  "STEP": 2,
  "VIRTUAL_ENTITY_1": {
    "VIRTUAL_ENTITY_ID": "V600601",
    "MEMBER_RECORDS": [
      {
        "INTERNAL_ID": 600601,
        "RECORDS": [
          {
            "DATA_SOURCE": "CA-CONTRACTORS",
            "RECORD_ID": "1078219"
          }
        ]
      }
    ]
  },
  "VIRTUAL_ENTITY_2": {
    "VIRTUAL_ENTITY_ID": "V600740",
    "MEMBER_RECORDS": [
      {
        "INTERNAL_ID": 600740,
        "RECORDS": [
          {
            "DATA_SOURCE": "CA-CONTRACTORS",
            "RECORD_ID": "1093103"
          }
        ]
      }
    ]
  },
  "INBOUND_VIRTUAL_ENTITY_ID": "V600601",
  "RESULT_VIRTUAL_ENTITY_ID": "V600601-S2",
  "MATCH_INFO": {
    "MATCH_KEY": "+LIKELY_NAME+ADDRESS+PHONE",
    "ERRULE_CODE": "MFF_LNAME_CEXCL",
    "FEATURE_SCORES": {
      "ADDRESS": [
        {
          "INBOUND_FEAT_ID": 813636,
          "INBOUND_FEAT_DESC": "11555 DUBLIN BOULEVARD DUBLIN CA 94568",
          "INBOUND_FEAT_USAGE_TYPE": "BUSINESS",
          "CANDIDATE_FEAT_ID":

In [52]:
types = pd.read_sql("""
    SELECT dsrc_id,
           count(*) FILTER (WHERE json_data::jsonb->'FEATURES' @> '[{"RECORD_TYPE":"PERSON"}]')       AS persons,
           count(*) FILTER (WHERE json_data::jsonb->'FEATURES' @> '[{"RECORD_TYPE":"ORGANIZATION"}]') AS orgs
    FROM dsrc_record GROUP BY dsrc_id ORDER BY 2 DESC
""", db)
types['data_source'] = types.dsrc_id.map(names)
print(types[['data_source', 'persons', 'orgs']].to_string(index=False))

            data_source  persons  orgs
CA-CONTRACTOR-PERSONNEL    10033   182
      CA-SOS-PRINCIPALS     5523   178
          CA-SOS-AGENTS     5039  2382
         CA-CONTRACTORS        0  3274
          ABC-NONRETAIL        0    19
         CA-SOS-FILINGS        0  7801
             ABC-RETAIL        0   201
             BURLINGAME        0  4781


In [53]:
person = json.loads(engine.get_entity_by_record_id(
    'CA-SOS-AGENTS', 'B20260328234', flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']

print(f"entity {person['ENTITY_ID']}: {person['ENTITY_NAME']}  ({len(person['RECORDS'])} records)")
print(f"  type:    {person['FEATURES']['RECORD_TYPE'][0]['FEAT_DESC']}")
print(f"  name:    {[n['FEAT_DESC'] for n in person['FEATURES']['NAME']]}")
print(f"  address: {person['FEATURES']['ADDRESS'][0]['FEAT_DESC']}")

entity 302070: ZHONGHAO CAO  (10 records)
  type:    PERSON
  name:    ['Zhonghao Cao']
  address: 828 W EL REPETTO DR MONTEREY PARK CA 91754 UNITED STATES


In [54]:
print(f"{person['ENTITY_NAME']} appears as:\n")
for r in person['RECORDS']:
    rec = json.loads(engine.get_record(r['DATA_SOURCE'], r['RECORD_ID']))['JSON_DATA']
    company = rec.get('COMPANY_NAME') or rec.get('REPRESENTS_COMPANY') or '?'
    role = rec.get('POSITION_TYPE') or rec.get('AGENT_TYPE') or ''
    print(f"  {company[:34]:<34} {role}")

ZHONGHAO CAO appears as:

  ZHC CONSTRUCTION INC               Individual Agent
  HE WINDOW INC                      Individual Agent
  ZHC CONSTRUCTION INC               Chief Executive Officer
  ZHC CONSTRUCTION INC               Chief Financial Officer
  ZHC CONSTRUCTION INC               Secretary
  ZHC CONSTRUCTION INC               Director
  HE WINDOW INC                      Chief Executive Officer
  HE WINDOW INC                      Chief Financial Officer
  HE WINDOW INC                      Secretary
  HE WINDOW INC                      Director


In [56]:
busiest = pd.read_sql("""
    SELECT k.res_ent_id AS entity_id,
           count(DISTINCT COALESCE(d.json_data::jsonb->>'COMPANY_NAME',
                                   d.json_data::jsonb->>'REPRESENTS_COMPANY')) AS companies,
           count(*) AS records
    FROM dsrc_record d
    JOIN obs_ent o      ON o.dsrc_id = d.dsrc_id AND o.ent_src_key = d.ent_src_key
    JOIN res_ent_okey k ON k.obs_ent_id = o.obs_ent_id
    WHERE d.json_data::jsonb->'FEATURES' @> '[{"RECORD_TYPE":"PERSON"}]'
    GROUP BY 1
    HAVING count(DISTINCT COALESCE(d.json_data::jsonb->>'COMPANY_NAME',
                                   d.json_data::jsonb->>'REPRESENTS_COMPANY')) > 1
    ORDER BY 2 DESC LIMIT 10
""", db)

for row in busiest.itertuples():
    e = json.loads(engine.get_entity_by_entity_id(row.entity_id))['RESOLVED_ENTITY']
    print(f"  {e['ENTITY_NAME'][:36]:<36} {row.companies} companies, {row.records} records")


  Shelene Nordstrom-Vaka               20 companies, 20 records
  Bhanu Partap Rana                    11 companies, 11 records
  SUSANNA WATT                         9 companies, 9 records
  Michael Kelly                        8 companies, 8 records
  David Fradin                         8 companies, 8 records
  KRISTINA PETROSYAN                   5 companies, 5 records
  Rachel Novak Siegel                  5 companies, 7 records
  Ishkhan Shahazizyan                  5 companies, 5 records
  Alejandro Duran                      4 companies, 4 records
  JAY K PAUL                           4 companies, 4 records


In [57]:
pd.read_sql("""
    SELECT round(latitude::numeric, 5)  AS lat,
           round(longitude::numeric, 5) AS lon,
           count(DISTINCT entity_id)    AS businesses,
           min(street)                  AS example_address
    FROM workshop.entity_geo
    GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 10
""", db)


,lat,lon,businesses,example_address
0,37.58992,-122.34032,118,533 AIRPORT BLVD
1,37.57830,-122.34867,71,1354 BURLINGAME AVE
2,37.59447,-122.38618,69,1533 PLAZA LN
3,37.57712,-122.34910,65,1400 BURLINGAME AVE
4,37.57898,-122.34570,63,1101 BURLINGAME AVE
5,37.59940,-122.36944,49,1633 BAYSHORE 136
6,37.57780,-122.34613,46,1220 HOWARD AVE # 216
7,37.58202,-122.34187,45,850 BURLINGAME AVE
8,37.57662,-122.34656,43,1321 HOWARD AVE
9,37.57982,-122.34675,41,1202 BURLINGAME AVE


In [58]:
results = json.loads(engine.search_by_attributes(
    json.dumps({'NAME_ORG': 'HYATT REGENCY SAN FRANCISCO AIRPORT'})))

for hit in results.get('RESOLVED_ENTITIES', []):
    e = hit['ENTITY']['RESOLVED_ENTITY']
    print(f"entity {e['ENTITY_ID']}: {e['ENTITY_NAME']}  match={hit['MATCH_INFO']['MATCH_KEY']}")


entity 2120: HYATT REGENCY SAN FRANCISCO AIRPORT  match=+NAME


In [59]:
pd.read_sql("""
    SELECT dsrc_id, record_id,
           json_data::jsonb->'FEATURES'->1->>'NAME_ORG' AS name
    FROM dsrc_record WHERE json_data ILIKE '%%HYATT%%'
    ORDER BY dsrc_id, record_id
""", db)


,dsrc_id,record_id,name
0,1001,20500020,HYATT REGENCY SAN FRANCISCO AIRPORT
1,1002,220571-47,HYATT REGENCY SAN FRANCISCO AIRPORT
2,1002,220571-58,HYATT REGENCY SAN FRANCISCO AIRPORT
3,1006,B20260328237-P2462,None
4,1006,B20260328237-P2463,None
5,1006,B20260328237-P2464,None
6,1006,B20260328237-P2465,None
7,1006,B20260328237-P2466,None
8,1006,B20260328237-P2467,None
9,1008,971138-120962,None


In [60]:
HYATT = 2120

e = json.loads(engine.get_entity_by_entity_id(
    HYATT, flags=F.SZ_ENTITY_DEFAULT_FLAGS))['RESOLVED_ENTITY']

print(f"{e['ENTITY_NAME']}  ({len(e['RECORDS'])} records)")
for r in e['RECORDS']:
    print(f"   {r['DATA_SOURCE']:<14} {r['RECORD_ID']}")
print(f"   {e['FEATURES']['ADDRESS'][0]['FEAT_DESC']}")


HYATT REGENCY SAN FRANCISCO AIRPORT  (3 records)
   BURLINGAME     20500020
   ABC-RETAIL     220571-47
   ABC-RETAIL     220571-58
   1333 BAYSHORE HWY, BURLINGAME, CA 94010-1886


In [61]:
why = json.loads(engine.why_records(
    'BURLINGAME', '20500020', 'ABC-RETAIL', '220571-47'))
info = why['WHY_RESULTS'][0]['MATCH_INFO']

print(f"{info['WHY_KEY']}  ->  {info['MATCH_LEVEL_CODE']}")
for family, scores in info['FEATURE_SCORES'].items():
    print(f"   {family:<14} {scores[0]['SCORE_BUCKET']}")


+NAME+ADDRESS  ->  RESOLVED
   ADDRESS        SAME
   NAME           SAME
   RECORD_TYPE    SAME


In [62]:
d = json.loads(engine.get_entity_by_entity_id(HYATT, flags=F.SZ_ENTITY_DEFAULT_FLAGS))

for r in d['RELATED_ENTITIES']:
    print(f"  {r['ENTITY_NAME'][:42]:<42} {r['MATCH_LEVEL_CODE']}  {r['MATCH_KEY']}")


  CURTIN CONVENTION & EXPOSITION SERVICE     POSSIBLY_RELATED  +ADDRESS
  ENCORE GROUP USA LLC                       POSSIBLY_RELATED  +ADDRESS
  HALLMARK MARKETING CORPORATION             POSSIBLY_RELATED  +ADDRESS
  IRON MOUNTAIN                              POSSIBLY_RELATED  +ADDRESS
  TRAVEL TRADERS #698                        POSSIBLY_RELATED  +ADDRESS


In [63]:
net = json.loads(engine.find_network_by_entity_id(
    [HYATT], 2, 1, 10, flags=F.SZ_FIND_NETWORK_DEFAULT_FLAGS))

print(f"{len(net['ENTITIES'])} entities in the network\n")
for e in net['ENTITIES']:
    r = e['RESOLVED_ENTITY']
    print(f"  {r['ENTITY_ID']:>6}  {r.get('ENTITY_NAME','')[:44]}")


6 entities in the network

    1259  CURTIN CONVENTION & EXPOSITION SERVICE
    1561  ENCORE GROUP USA LLC
    1993  HALLMARK MARKETING CORPORATION
    2120  HYATT REGENCY SAN FRANCISCO AIRPORT
    2202  IRON MOUNTAIN
    4394  TRAVEL TRADERS #698


In [64]:
nearby = pd.read_sql("""
    WITH me AS (SELECT latitude la, longitude lo
                FROM workshop.entity_geo WHERE entity_id = 2120 LIMIT 1)
    SELECT DISTINCT g.entity_id, g.street,
           round((6371000*2*asin(sqrt(
               power(sin(radians(g.latitude - me.la)/2), 2) +
               cos(radians(me.la))*cos(radians(g.latitude)) *
               power(sin(radians(g.longitude - me.lo)/2), 2))))::numeric) AS metres
    FROM workshop.entity_geo g, me
    WHERE g.entity_id <> 2120 AND g.street NOT LIKE 'PO BOX%%'
    ORDER BY metres LIMIT 20
""", db)

for row in nearby.itertuples():
    e = json.loads(engine.get_entity_by_entity_id(row.entity_id))['RESOLVED_ENTITY']
    print(f"  {row.metres:>5.0f}m  {e['ENTITY_NAME'][:44]}")


      0m  CURTIN CONVENTION & EXPOSITION SERVICE
      0m  ENCORE GROUP USA LLC
      0m  HALLMARK MARKETING CORPORATION
      0m  IRON MOUNTAIN
      0m  TRAVEL TRADERS #698
    131m  B Y & S CPAS A PROFESSIONAL CORPORATION
    131m  BAY AREA SERVICE
    131m  CAL MOBILITY
    131m  E KWONG ACUPUNCTURE CLINIC
    131m  EASA ARCHITECTURE
    131m  ELIE CORP
    131m  ENERGY REAL ESTATE INC
    131m  ER STUDIO
    131m  FARHAD TAX BOOKKEEPING CONSULTING SERV
    131m  FEEDLITECH
    131m  GREEN CAB PENINSULA
    131m  GUARANTEE MORTGAGE
    131m  HABITAT
    131m  HABITAT
    131m  INTENTIONAL HEALTH


In [65]:
sql = """
WITH me AS (SELECT latitude la, longitude lo
            FROM workshop.entity_geo WHERE entity_id = 2120 LIMIT 1),
abc AS (
  SELECT k.res_ent_id AS entity_id,
         string_agg(DISTINCT d.json_data::jsonb->>'LICENSE_TYPE', ',')   AS lic_types,
         string_agg(DISTINCT d.json_data::jsonb->>'LICENSE_STATUS', ',') AS status
  FROM dsrc_record d
  JOIN obs_ent o      ON o.dsrc_id = d.dsrc_id AND o.ent_src_key = d.ent_src_key
  JOIN res_ent_okey k ON k.obs_ent_id = o.obs_ent_id
  WHERE d.dsrc_id IN (1002, 1003)          -- ABC-RETAIL, ABC-NONRETAIL
  GROUP BY 1)
SELECT DISTINCT a.entity_id, g.street, a.lic_types, a.status,
   round((6371000*2*asin(sqrt(
       power(sin(radians(g.latitude - me.la)/2), 2) +
       cos(radians(me.la))*cos(radians(g.latitude)) *
       power(sin(radians(g.longitude - me.lo)/2), 2))))::numeric) AS metres
FROM abc a JOIN workshop.entity_geo g ON g.entity_id = a.entity_id, me
WHERE a.entity_id <> 2120
ORDER BY metres LIMIT 20
"""

for r in pd.read_sql(sql, db).itertuples():
    e = json.loads(engine.get_entity_by_entity_id(r.entity_id))['RESOLVED_ENTITY']
    print(f"  {r.metres:>5.0f}m (~{int(r.metres/80):>2d} min)  "
          f"{e['ENTITY_NAME'][:34]:<34} type {r.lic_types:<6} {r.street[:26]}")


    131m (~ 1 min)  MR TERIYAKI                        type 41     851 BURLWAY RD # 900
    131m (~ 1 min)  MR TERIYAKI                        type 41     851 BURLWAY RD, STE 900
    263m (~ 3 min)  MAXS OF BURLINGAME                 type 47,58  1250 BAYSHORE HWY
    355m (~ 4 min)  GRAND HARBOR SEAFOOD AND DIMSUM    type 47     1492 BAYSHORE HWY
    409m (~ 5 min)  GRAND HARBOR SEAFOOD AND DIMSUM    type 47     1492 OLD BAYSHORE HWY
    478m (~ 5 min)  LEGACY THROUGH GIVING FOUNDATION   type 17     1321 N CAROLAN AVE
    486m (~ 6 min)  BENIHANA OF TOKYO INC              type 47     1496 BAYSHORE HWY
    577m (~ 7 min)  H L PENINSULA PEARL                type 47     1590 BAYSHORE HWY
    656m (~ 8 min)  CROWNE PLAZA SAN FRANCISCO AIRPORT type 47     1177 AIRPORT BLVD
    692m (~ 8 min)  H L PENINSULA HOT POT & GRILL LLC  type 47     1600 BAYSHORE HWY
    692m (~ 8 min)  MAVERICK JACKS                     type 41,58  1190 CALIFORNIA DR
    699m (~ 8 min)  VAGABOND INN EXECUTIVE SAN FRA